# 🔬 NB3 · End-to-End Worked Example: "100% accuracy" · Caso completo: "100% de exactitud"
**Research Methods & Scientific Integrity in AI · Session 5 · UNMSM Deep Tech Doctorate · Dr. Loveleen Gaur**  
*Métodos de Investigación e Integridad Científica en IA · Sesión 5*

**EN:** A reproduction audit from claim to honest report, in five acts. Instructor walk-through in Block 5 (≈ 10 min); students re-run it afterwards.  
**ES:** Una auditoría de reproducción, de la afirmación al informe honesto, en cinco actos.

> ⚠️ **Teaching scenario / Escenario didáctico:** the "paper" is fictional; the dataset, code and every number below are real and re-computed live.  
> Dataset: Wisconsin Diagnostic Breast Cancer (569 patients, 30 features), shipped with scikit-learn ([UCI record](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)).

| Act | EN | ES |
|---|---|---|
| 1 | The claim | La afirmación |
| 2 | Reproduce it exactly | Reproducirla tal cual |
| 3 | Audit: find the flaws | Auditar: encontrar los fallos |
| 4 | Fix and re-run honestly | Corregir y volver a ejecutar |
| 5 | Report like a scientist | Reportar como científico |

> **🇪🇸 Traducción al español**
>
> # 🔬 NB3 · Ejemplo completo de principio a fin: "100 % de exactitud"
> **Métodos de Investigación e Integridad Científica en IA · Sesión 5 · Doctorado en Deep Tech UNMSM · Dra. Loveleen Gaur**
>
> Una auditoría de reproducción, de la afirmación al informe honesto, en cinco actos. Recorrido guiado del docente en el Bloque 5 (≈ 10 min); los estudiantes lo vuelven a ejecutar después.
>
> ⚠️ **Escenario didáctico:** el "artículo" es ficticio; el conjunto de datos, el código y cada número de abajo son reales y se recalculan en vivo.
> Conjunto de datos: Wisconsin Diagnostic Breast Cancer (569 pacientes, 30 variables), incluido en scikit-learn ([registro UCI](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)).
>
> | Acto | Descripción |
> |---|---|
> | 1 | La afirmación |
> | 2 | Reproducirla tal cual |
> | 3 | Auditar: encontrar los fallos |
> | 4 | Corregir y volver a ejecutar con honestidad |
> | 5 | Reportar como un científico |

In [ ]:
# Importa módulos del sistema, plataforma, advertencias, NumPy, pandas, scikit-learn y SciPy
import sys, platform, warnings, numpy as np, pandas as pd, sklearn, scipy
# Silencia las advertencias
warnings.filterwarnings("ignore")
# Carga el conjunto de datos de cáncer de mama
from sklearn.datasets import load_breast_cancer
# División entrenamiento/prueba, validación cruzada estratificada repetida y su evaluación
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score
# Escalador que estandariza las variables
from sklearn.preprocessing import StandardScaler
# Selección de las k mejores variables con la prueba F
from sklearn.feature_selection import SelectKBest, f_classif
# Red neuronal multicapa
from sklearn.neural_network import MLPClassifier
# Regresión logística (línea base simple)
from sklearn.linear_model import LogisticRegression
# Clasificador trivial que siempre predice la clase mayoritaria
from sklearn.dummy import DummyClassifier
# Encadena pasos en un Pipeline
from sklearn.pipeline import make_pipeline
# Métrica de exactitud
from sklearn.metrics import accuracy_score
# Imprime las versiones de Python, scikit-learn y NumPy
print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__, "| NumPy", np.__version__)
# Carga variables (X) y etiquetas (y); 0 = maligno, 1 = benigno
X, y = load_breast_cancer(return_X_y=True)
# Imprime número de pacientes, variables, malignos y benignos
print("Patients:", X.shape[0], "| Features:", X.shape[1], "| Malignant:", (y==0).sum(), "| Benign:", (y==1).sum())

## Act 1 · The claim · La afirmación
> *"Our deep neural network (DeepOnco-MLP) detects malignant tumours with **100.0% accuracy**, outperforming prior work. We balanced the classes by oversampling, standardised and selected the 10 most informative features, and evaluated on a held-out 20% test set (seed 12)."*

**EN:** Sounds rigorous: balancing, feature selection, held-out test set, a fixed seed. Poll: do you believe it?  
**ES:** Suena riguroso. Encuesta: ¿lo crees?

> **🇪🇸 Traducción al español**
>
> ## Acto 1 · La afirmación
> > *"Nuestra red neuronal profunda (DeepOnco-MLP) detecta tumores malignos con **100.0 % de exactitud**, superando trabajos previos. Equilibramos las clases mediante sobremuestreo, estandarizamos y seleccionamos las 10 variables más informativas, y evaluamos con un conjunto de prueba reservado del 20 % (semilla 12)."*
>
> Suena riguroso: equilibrio de clases, selección de variables, conjunto de prueba reservado y semilla fija. Encuesta: ¿lo crees?

In [ ]:
# Reproduce el método del "artículo" con una semilla dada y devuelve la exactitud
def paper_pipeline(seed):
    # Generador aleatorio reproducible a partir de la semilla
    rng = np.random.default_rng(seed)
    # 1) oversample the minority class to balance classes ... BEFORE splitting
    # ES: 1) sobremuestrear la clase minoritaria para equilibrar las clases ... ANTES de dividir
    # Índices de los pacientes malignos (clase minoritaria)
    minority = np.where(y == 0)[0]
    # Sortea con reemplazo los malignos que faltan para igualar a los benignos
    extra = rng.choice(minority, (y == 1).sum() - len(minority), replace=True)
    # Añade las copias: Xo (variables) y yo (etiquetas) sobremuestreados
    Xo, yo = np.vstack([X, X[extra]]), np.concatenate([y, y[extra]])
    # 2) standardise and select features on ALL rows ... BEFORE splitting
    # ES: 2) estandarizar y seleccionar variables con TODAS las filas ... ANTES de dividir
    # Estandariza con todas las filas (fuga de datos)
    Xo = StandardScaler().fit_transform(Xo)
    # Selecciona las 10 mejores variables con todas las filas (fuga de datos)
    Xo = SelectKBest(f_classif, k=10).fit_transform(Xo, yo)
    # 3) one 80/20 split, one seed
    # ES: 3) una sola división 80/20, una sola semilla
    Xtr, Xte, ytr, yte = train_test_split(Xo, yo, test_size=0.2, random_state=seed)
    # Entrena la red neuronal (64 y 32 neuronas) con la misma semilla
    m = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=2000, random_state=seed).fit(Xtr, ytr)
    # Devuelve la exactitud sobre el conjunto de prueba
    return accuracy_score(yte, m.predict(Xte))

# Reproduce el resultado con la semilla 12 del artículo
print(f"Act 2 · Reproduced with the paper's seed 12: {paper_pipeline(12):.1%}")

## Act 2 · It reproduces! · ¡Se reproduce!
**EN:** Same code, same data, same seed, same number. The result is **reproducible**. Is it **true**? Reproducibility is the receipt, not the proof.  
**ES:** Mismo código, datos y semilla: mismo número. Es **reproducible**. ¿Es **verdadero**?

## Act 3 · The audit · La auditoría
**EN:** Three questions a reviewer should ask. (a) Was seed 12 chosen *after* looking? (b) Can a row appear in both train and test? (c) Did any statistic see the test rows?  
**ES:** (a) ¿Se eligió la semilla 12 *después* de mirar? (b) ¿Puede una fila estar en entrenamiento y prueba? (c) ¿Alguna estadística vio las filas de prueba?

> **🇪🇸 Traducción al español**
>
> ## Acto 2 · ¡Se reproduce!
> Mismo código, mismos datos, misma semilla, mismo número. El resultado es **reproducible**. ¿Es **verdadero**? La reproducibilidad es el recibo, no la prueba.
>
> ## Acto 3 · La auditoría
> Tres preguntas que debería hacer un revisor. (a) ¿Se eligió la semilla 12 *después* de mirar los resultados? (b) ¿Puede una fila aparecer tanto en entrenamiento como en prueba? (c) ¿Alguna estadística vio las filas de prueba?

In [ ]:
# (a) The seed lottery: run the paper's exact pipeline with 20 seeds
# ES: (a) La lotería de semillas: ejecutar el pipeline exacto del artículo con 20 semillas
# Exactitud de cada semilla 0-19 en una Serie (índice = semilla)
res = pd.Series({s: paper_pipeline(s) for s in range(20)})
# Imprime las 20 exactitudes con 4 decimales
print(res.round(4).to_string())
# Indica si la semilla 12 es el máximo, y muestra la media de 20 semillas y el peor caso
print(f"\nSeed 12 is the maximum: {res.idxmax() == 12}.  Mean over 20 seeds: {res.mean():.1%}   Worst: {res.min():.1%}")

In [ ]:
# (b) Duplicates across the split: oversampling copies malignant patients BEFORE the split
# ES: (b) Duplicados entre las particiones: el sobremuestreo copia pacientes malignos ANTES de dividir
# Repite el sorteo de copias con la semilla 12
rng = np.random.default_rng(12)
# Índices de los pacientes malignos
minority = np.where(y == 0)[0]
# Pacientes malignos copiados con reemplazo
extra = rng.choice(minority, (y == 1).sum() - len(minority), replace=True)
# Identificador del paciente real detrás de cada fila (originales + copias)
patient_id = np.concatenate([np.arange(len(y)), extra])        # which real patient each row came from
# Divide los índices 80/20 igual que el artículo
tr_idx, te_idx = train_test_split(np.arange(len(patient_id)), test_size=0.2, random_state=12)
# Pacientes presentes en ambas particiones
shared = set(patient_id[tr_idx]) & set(patient_id[te_idx])
# Imprime filas de prueba, cuántas tienen al mismo paciente en entrenamiento y su porcentaje
print(f"Test rows: {len(te_idx)} | test rows whose patient ALSO appears in training: "
      # Continuación del print: cuenta las filas de prueba duplicadas y su proporción
      f"{sum(p in shared for p in patient_id[te_idx])} ({np.mean([p in shared for p in patient_id[te_idx]]):.0%})")

**(c)** Scaling and `SelectKBest` were fitted on all rows, including the future test rows (leakage types "pre-processing" and "feature selection on train and test").

**Verdict / Veredicto:** three flaws stacked: *hero seed* + *duplicates across the split* + *preprocessing leakage*.

## Act 4 · Fix and re-run honestly · Corregir y volver a ejecutar
**EN:** Everything that learns from data goes inside a `Pipeline`; oversampling (if used) only inside training folds; 5-fold cross-validation repeated 10 times; and always a **simple baseline** plus a **trivial baseline**.  
**ES:** Todo lo que aprende de los datos va dentro de un `Pipeline`; validación cruzada 5×10; y siempre una **línea base simple** y una **trivial**.

> **🇪🇸 Traducción al español**
>
> **(c)** El escalado y `SelectKBest` se ajustaron con todas las filas, incluidas las futuras filas de prueba (tipos de fuga: "preprocesamiento" y "selección de variables sobre entrenamiento y prueba").
>
> **Veredicto:** tres fallos apilados: *semilla héroe* + *duplicados entre particiones* + *fuga por preprocesamiento*.
>
> ## Acto 4 · Corregir y volver a ejecutar con honestidad
> Todo lo que aprende de los datos va dentro de un `Pipeline`; el sobremuestreo (si se usa) solo dentro de las particiones de entrenamiento; validación cruzada de 5 particiones repetida 10 veces; y siempre una **línea base simple** más una **línea base trivial**.

In [ ]:
# Validación cruzada 5 particiones x 10 repeticiones (50 pruebas) con semilla fija
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=2026)
# Modelos a comparar; todo lo que aprende va dentro de un Pipeline
models = {
    # Línea base trivial: siempre predice la clase mayoritaria
    "Trivial (majority class)": DummyClassifier(strategy="most_frequent"),
    # Línea base simple: regresión logística con escalado
    "Logistic regression (1958-era baseline)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    # MLP con pipeline honesto: escalado + selección de 10 variables + red neuronal
    "DeepOnco-MLP, honest pipeline": make_pipeline(StandardScaler(), SelectKBest(f_classif, k=10),
                                                   # Último paso del Pipeline: red neuronal (64 y 32 neuronas) con semilla fija; cierra make_pipeline y la entrada del diccionario
                                                   MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=2000, random_state=0)),
# Cierra el diccionario de modelos
}
# Acumula una fila de resultados por modelo
rows = []
# Evalúa cada modelo con las mismas particiones
for name, m in models.items():
    # Las 50 exactitudes
    s = cross_val_score(m, X, y, cv=cv)
    # Guarda media, desviación estándar, mínimo, máximo y número de particiones
    rows.append({"Model": name, "Mean accuracy": s.mean(), "SD": s.std(ddof=1), "Min": s.min(), "Max": s.max(), "Folds": len(s)})
# Tabla de resultados indexada por modelo
report = pd.DataFrame(rows).set_index("Model")
# Imprime la tabla con 4 decimales
print(report.round(4).to_string())

In [ ]:
# Importa matplotlib para graficar
import matplotlib.pyplot as plt
# Figura de 8x3.2 pulgadas con un eje
fig, ax = plt.subplots(figsize=(8, 3.2))
# Etiquetas de las cinco barras
labels = ["Paper claim\n(seed 12)", "Paper pipeline\n(mean of 20 seeds)", "MLP honest\n(5x10 CV)", "Logistic reg.\n(5x10 CV)", "Majority class"]
# Valores: afirmación, media del método del artículo, MLP honesto, regresión logística y clase mayoritaria
vals = [res[12], res.mean(), report.iloc[2, 0], report.iloc[1, 0], report.iloc[0, 0]]
# Un color por barra
cols = ["#E07A1F", "#F2B179", "#0E9F9A", "#0B1F3A", "#8A9AAD"]
# Dibuja las barras en porcentaje
ax.bar(labels, np.array(vals) * 100, color=cols)
# Escribe el valor sobre cada barra
for i, v in enumerate(vals): ax.text(i, v * 100 + 0.8, f"{v:.1%}", ha="center", fontsize=9)
# Límites del eje Y, título del eje Y y título del gráfico
ax.set_ylim(50, 105); ax.set_ylabel("Accuracy (%)"); ax.set_title("From claim to honest estimate · De la afirmación a la estimación honesta")
# Ajusta el diseño y muestra la figura
plt.tight_layout(); plt.show()

## Act 5 · Report like a scientist · Reportar como científico
**EN:** Note two things. The honest MLP is still good, but **plain logistic regression is at least as good**, which is exactly Kapoor & Narayanan's finding in civil-war prediction. And fold scores from repeated CV are not independent, so we report mean ± SD and avoid over-precise confidence intervals (see Session 6 for corrected tests).  
**ES:** El MLP honesto sigue siendo bueno, pero **la regresión logística es al menos igual de buena**. Los resultados de CV repetida no son independientes: reportamos media ± DE.

> **🇪🇸 Traducción al español**
>
> ## Acto 5 · Reportar como un científico
> Observa dos cosas. El MLP honesto sigue siendo bueno, pero **la regresión logística simple es al menos igual de buena**, que es justo el hallazgo de Kapoor y Narayanan en la predicción de guerras civiles. Además, las puntuaciones de las particiones de una validación cruzada repetida no son independientes, así que reportamos media ± DE y evitamos intervalos de confianza demasiado precisos (ver la Sesión 6 para pruebas corregidas).

In [ ]:
# Resultados de la línea base simple y del MLP honesto, buscados por nombre en la tabla
best_simple = report.loc["Logistic regression (1958-era baseline)"]
# Fila de resultados del MLP honesto, buscada por nombre en la tabla
mlp = report.loc["DeepOnco-MLP, honest pipeline"]
# Encabezado y separador
print("REPRODUCTION REPORT · INFORME DE REPRODUCCIÓN")
# Imprime una línea separadora de 60 guiones
print("-" * 60)
# ¿Se reprodujo la afirmación exactamente con la semilla 12?
print(f"Claim reproduced exactly (seed 12)      : YES, {res[12]:.1%}")
# ¿Sobrevive la afirmación a la auditoría? No
print(f"Claim survives audit                    : NO")
# Fallos encontrados
print(f"Flaws found                             : hero seed; duplicates across split; preprocessing leakage")
# Estimación honesta del MLP (media ± DE, CV 5x10)
print(f"Honest MLP estimate                     : {mlp['Mean accuracy']:.1%} ± {mlp['SD']:.1%} (5x10 CV)")
# Línea base simple (regresión logística)
print(f"Simple baseline (logistic regression)   : {best_simple['Mean accuracy']:.1%} ± {best_simple['SD']:.1%}")
# Entorno de ejecución
print(f"Environment                             : Python {platform.python_version()}, scikit-learn {sklearn.__version__}")
# Separador y conclusión
print("-" * 60)
# Conclusión del informe: el resultado era reproducible pero no válido
print("EN: The result was reproducible but not valid.  ES: El resultado era reproducible pero no válido.")

## 🎯 Your turn · Tu turno (homework / tarea)
1. **EN:** Move the oversampling *inside* the training folds (hint: `imblearn.pipeline.Pipeline` with `RandomOverSampler`). Does the MLP change? · **ES:** Mueve el sobremuestreo *dentro* de las particiones de entrenamiento.
2. Replace the MLP with your own thesis model. Which baseline would a reviewer demand? · Reemplaza el MLP con tu modelo de tesis.
3. Write the three-line *Reproduction Report* above for one published paper in your field. · Escribe el informe de tres líneas para un artículo de tu área.

**Further reading:** Kapoor & Narayanan (2023) [doi.org/10.1016/j.patter.2023.100804](https://doi.org/10.1016/j.patter.2023.100804) · REFORMS checklist, Kapoor et al. (2024) [doi.org/10.1126/sciadv.adk3452](https://doi.org/10.1126/sciadv.adk3452)

> **🇪🇸 Traducción al español**
>
> ## 🎯 Tu turno (tarea)
> 1. Mueve el sobremuestreo *dentro* de las particiones de entrenamiento (pista: `imblearn.pipeline.Pipeline` con `RandomOverSampler`). ¿Cambia el MLP?
> 2. Reemplaza el MLP por el modelo de tu tesis. ¿Qué línea base exigiría un revisor?
> 3. Escribe el *Informe de reproducción* de tres líneas de arriba para un artículo publicado en tu área.
>
> **Lecturas adicionales:** Kapoor & Narayanan (2023) [doi.org/10.1016/j.patter.2023.100804](https://doi.org/10.1016/j.patter.2023.100804) · lista de verificación REFORMS, Kapoor et al. (2024) [doi.org/10.1126/sciadv.adk3452](https://doi.org/10.1126/sciadv.adk3452)